# Day 2 AM — Frameworks, Isolation, Reliability, and HITL

Run this notebook top-to-bottom. Domain facts come only from canonical Week 2 fixtures (for example `Week2/data/insurance/workflow_cases_15.json`). Never invent underwriting or claim records.

**Before §01:** complete the primers in order — [`_CREWAI_PRIMER.ipynb`](_CREWAI_PRIMER.ipynb), [`_GOOGLE_ADK_PRIMER.ipynb`](_GOOGLE_ADK_PRIMER.ipynb), [`_MICROSOFT_AGENT_FRAMEWORK_PRIMER.ipynb`](_MICROSOFT_AGENT_FRAMEWORK_PRIMER.ipynb). Theory reference: [`_CONCEPTS.ipynb`](_CONCEPTS.ipynb). Progressive terminal scripts remain in [`_README.html`](_README.html).

Two **Exercises** appear near the end (after §06). Attempt them before opening [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb).

Later cells may redefine names. Restart the kernel before jumping backward mid-notebook. Read-only model/API calls run after local preflight. Cells that persist audit, checkpoint, or ledger data stay disabled until you set explicit flags.

In [ ]:
print()

In [ ]:
# AM shared setup — notebook-safe paths and configuration
import json
import os
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from dotenv import load_dotenv

_candidates = [Path.cwd(), *Path.cwd().parents]
WEEK2_ROOT = next(
    (p for p in _candidates if p.name == "Week2" and (p / "data").is_dir()),
    next((p / "Week2" for p in _candidates if (p / "Week2" / "data").is_dir()), None),
)
if WEEK2_ROOT is None:
    raise RuntimeError("Could not locate Week2/data from the current notebook working directory.")
SESSION_DIR = WEEK2_ROOT / "W2D2" / "W2D2AM"
DATA_PATH = WEEK2_ROOT / "data" / "insurance" / "workflow_cases_15.json"
OUTPUT_DIR = SESSION_DIR / "outputs"
load_dotenv(WEEK2_ROOT / ".env")

CASE_ID = "UW-001"
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
REQUEST_TIMEOUT_SECONDS = float(os.getenv("AGENT_REQUEST_TIMEOUT_SECONDS", "60"))
print({"stage": "AM-SETUP", "week2_root": str(WEEK2_ROOT), "data_path": str(DATA_PATH)})

## 01 Framework landscape and evidence-based selection

**Required before the AM session:** complete the primers in this order: [CrewAI 1.15.x](_CREWAI_PRIMER.ipynb), [Google ADK 2.x](_GOOGLE_ADK_PRIMER.ipynb), then [Microsoft Agent Framework 1.x](_MICROSOFT_AGENT_FRAMEWORK_PRIMER.ipynb). They establish each framework's real model, tool, state/session, orchestration, safeguard, observability, evaluation, and deployment APIs before the comparative lesson.

Framework choice starts with workload requirements and auditable API evidence, not a feature-count score. CrewAI separates role-based `Agent`/`Task`/`Crew` collaboration from event-driven typed `Flow` control. Google ADK uses `Agent` (`LlmAgent`), function tools, `Runner`, session services, callbacks, and workflow agents. Microsoft Agent Framework uses `Agent`, `FoundryChatClient`, framework-native tools, `AgentSession`, orchestration builders, checkpoint storage, and approval events. LangGraph exposes explicit `StateGraph` nodes and edges, thread-scoped checkpointers, `interrupt()`, and `Command(resume=...)`. Claude Agent SDK exposes `ClaudeAgentOptions`, `AgentDefinition`, MCP tools, permission callbacks, lifecycle hooks, turn limits, and budget limits.

The tradeoff is control versus abstraction and operational fit: explicit graphs make state transitions and recovery easy to inspect but require more orchestration code; role-oriented frameworks accelerate specialist collaboration but can hide coordination cost; provider-oriented runtimes integrate identity and deployment well but increase service dependencies; SDK subagents provide strong context/tool scoping but require careful permission and lifecycle design. Installed-package availability is a deployment constraint, not proof that a framework meets a requirement.

<img src="Images/d2am_t1_framework_landscape.png" width="900" alt="Framework landscape comparing agent frameworks by state, permissions, isolation, coordination, and deterministic operation">

The cells below first build a validated evidence map, then add a provider-backed structured recommendation and local citation validation.


In [ ]:
# AM1.1 — source-linked framework evidence
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

FrameworkName = Literal["CrewAI", "Google ADK", "Microsoft Agent Framework", "LangGraph", "Claude Agent SDK"]

class EvidenceItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    evidence_id: str
    framework: FrameworkName
    capability: Literal["workflow_control", "durable_state", "multi_agent", "tool_boundary", "human_approval", "session_runtime"]
    observed_surface: str
    source_url: str

class FrameworkEvidence(BaseModel):
    model_config = ConfigDict(extra="forbid")
    framework: FrameworkName
    package_name: str
    installed_version: str | None
    evidence: list[EvidenceItem]

PACKAGE_NAMES: dict[FrameworkName, str] = {
    "CrewAI": "crewai", "Google ADK": "google-adk", "Microsoft Agent Framework": "agent-framework",
    "LangGraph": "langgraph", "Claude Agent SDK": "claude-agent-sdk",
}
RAW_EVIDENCE = [
    {"evidence_id":"crewai-crews-flows","framework":"CrewAI","capability":"multi_agent","observed_surface":"Crews coordinate role-based agents; Flows provide event-driven control.","source_url":"https://docs.crewai.com/en/concepts/crews"},
    {"evidence_id":"google-adk-workflow-agents","framework":"Google ADK","capability":"workflow_control","observed_surface":"Sequential, parallel, and loop workflow agents orchestrate sub-agents.","source_url":"https://google.github.io/adk-docs/agents/workflow-agents/"},
    {"evidence_id":"msaf-workflows","framework":"Microsoft Agent Framework","capability":"workflow_control","observed_surface":"Graph workflows connect typed executors and edges.","source_url":"https://learn.microsoft.com/en-us/agent-framework/user-guide/workflows/"},
    {"evidence_id":"langgraph-persistence","framework":"LangGraph","capability":"durable_state","observed_surface":"Checkpointers persist graph state by thread for pause and resume.","source_url":"https://docs.langchain.com/oss/python/langgraph/persistence"},
    {"evidence_id":"langgraph-graph-api","framework":"LangGraph","capability":"workflow_control","observed_surface":"StateGraph declares nodes, edges, and conditional routing explicitly.","source_url":"https://docs.langchain.com/oss/python/langgraph/graph-api"},
    {"evidence_id":"langgraph-interrupts","framework":"LangGraph","capability":"human_approval","observed_surface":"Interrupts pause execution and resume with external input.","source_url":"https://docs.langchain.com/oss/python/langgraph/interrupts"},
    {"evidence_id":"claude-sdk-subagents","framework":"Claude Agent SDK","capability":"multi_agent","observed_surface":"AgentDefinition gives a worker fresh context and an explicit tool scope.","source_url":"https://code.claude.com/docs/en/agent-sdk/subagents"},
    {"evidence_id":"claude-sdk-options","framework":"Claude Agent SDK","capability":"session_runtime","observed_surface":"Options bound tools, turns, budget, cwd, settings, and MCP servers.","source_url":"https://code.claude.com/docs/en/agent-sdk/python"},
]

def installed_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

def build_evidence_map() -> list[FrameworkEvidence]:
    validated = [EvidenceItem.model_validate(item) for item in RAW_EVIDENCE]
    result = [FrameworkEvidence(framework=f, package_name=p, installed_version=installed_version(p), evidence=[e for e in validated if e.framework == f]) for f, p in PACKAGE_NAMES.items()]
    if any(not entry.evidence for entry in result):
        raise RuntimeError("Every framework must have at least one evidence item.")
    return result

evidence_map = build_evidence_map()
print(json.dumps({"stage":"AM1.1", "frameworks":[e.model_dump() for e in evidence_map]}, indent=2))

In [ ]:
# AM1.2 — Anthropic structured recommendation over supplied evidence
import anthropic

class RequirementProfile(BaseModel):
    model_config = ConfigDict(extra="forbid")
    durable_resume: bool
    human_approval: bool
    isolated_workers: bool
    prefer_explicit_graph: bool
    require_installed_package: bool

class RequirementMatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    requirement: Literal["durable_resume", "human_approval", "isolated_workers", "prefer_explicit_graph"]
    evidence_id: str

class FrameworkRecommendation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    selected_framework: FrameworkName
    matches: list[RequirementMatch] = Field(min_length=1)
    uncovered_requirements: list[Literal["durable_resume", "human_approval", "isolated_workers", "prefer_explicit_graph"]]
    confidence: float = Field(ge=0, le=1)

def validate_recommendation(rec: FrameworkRecommendation, requirements: RequirementProfile) -> None:
    selected = next(item for item in evidence_map if item.framework == rec.selected_framework)
    if requirements.require_installed_package and selected.installed_version is None:
        raise ValueError("Selected framework package is not installed.")
    evidence = {item.evidence_id: item for item in selected.evidence}
    expected = {"durable_resume":"durable_state", "human_approval":"human_approval", "isolated_workers":"multi_agent", "prefer_explicit_graph":"workflow_control"}
    active = {name for name in expected if getattr(requirements, name)}
    covered = set()
    for match in rec.matches:
        if match.requirement not in active or match.evidence_id not in evidence:
            raise ValueError("Recommendation cited inactive, unknown, or cross-framework evidence.")
        if evidence[match.evidence_id].capability != expected[match.requirement]:
            raise ValueError("Evidence capability does not support the matched requirement.")
        covered.add(match.requirement)
    uncovered = set(rec.uncovered_requirements)
    if covered & uncovered or covered | uncovered != active:
        raise ValueError("Every active requirement must be matched or explicitly uncovered.")

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("AM1 preflight blocked: set ANTHROPIC_API_KEY in Week2/.env.")
requirements = RequirementProfile(durable_resume=True, human_approval=True, isolated_workers=False, prefer_explicit_graph=True, require_installed_package=True)
prompt = (
    "Recommend exactly one framework using only this evidence. Match every active requirement to an evidence_id from the selected framework, or list it as uncovered. Do not infer unlisted capabilities.\n"
    f"Requirements: {requirements.model_dump_json()}\nEvidence: {json.dumps([x.model_dump() for x in evidence_map])}"
)
started = time.perf_counter()
result = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=REQUEST_TIMEOUT_SECONDS, max_retries=1).messages.parse(
    model=MODEL, max_tokens=min(int(os.getenv("AGENT_MAX_TOKENS", "8000")), 2000), temperature=0,
    messages=[{"role":"user", "content":prompt}], output_format=FrameworkRecommendation,
)
if result.parsed_output is None:
    raise RuntimeError("Anthropic returned no parsed FrameworkRecommendation.")
validate_recommendation(result.parsed_output, requirements)
print(json.dumps({"stage":"AM1.2", "recommendation":result.parsed_output.model_dump(), "usage":result.usage.model_dump(), "stop_reason":result.stop_reason, "latency_seconds":round(time.perf_counter()-started,3)}, indent=2))

## 02 Scaffold reliability and typed provider boundaries

A reliable scaffold is the application control plane around a model call. Before spending tokens it verifies runtime packages, credential presence, canonical data existence and schema, unique identifiers, and configuration. At the call boundary it supplies narrow instructions, deterministic temperature, token and timeout limits, a Pydantic output contract, and explicit error envelopes. After the call it validates identity-critical fields against the source and records model, usage, stop reason, and wall-clock latency.

Anthropic's Python SDK `messages.parse(..., output_format=CaseBrief)` asks the provider for structured output and parses it into a Pydantic model. This guarantees shape, not factual truth: a separate local validator must reject changed `case_id` or `recommended_action`. `timeout` bounds waiting, `max_retries` bounds SDK transport retries, and `max_tokens` bounds one response; none replaces workflow-level budgets or domain authorization.

The main tradeoff is strictness versus flexibility. Fail-fast readiness checks make configuration errors visible and avoid paid calls, but every required dependency must be maintained accurately. Typed output reduces parsing ambiguity, but schema-valid hallucinations remain possible. Returning an explicit success/error report improves observability, while re-raising the final failure keeps automation from mistaking an error envelope for success.

<img src="Images/d2am_t2_scaffold_reliability.png" width="900" alt="Reliable agent scaffold with fail-fast setup, typed provider boundary, grounding validation, and observable outcomes">

The next cells establish typed canonical data and readiness, then cross the provider boundary only when that report is ready.


In [ ]:
# AM2.1 — typed canonical data and readiness preflight
class WorkflowCase(BaseModel):
    model_config = ConfigDict(extra="forbid")
    case_id: str = Field(pattern=r"^UW-[0-9]{3}$")
    applicant: str
    product: str
    requested_limit_usd: int = Field(gt=0)
    risk_flags: list[str]
    documents_complete: bool
    recommended_action: Literal["auto_approve", "refer_underwriter", "request_documents"]

class WorkflowDataset(BaseModel):
    model_config = ConfigDict(extra="forbid")
    cases: list[WorkflowCase]

class ReadinessReport(BaseModel):
    status: Literal["ready", "blocked"]
    python_version: str
    packages: list[dict]
    credentials: list[dict]
    dataset: dict
    errors: list[str]

PACKAGE_REQUIREMENTS = (("anthropic", True), ("claude-agent-sdk", True), ("pydantic", True), ("python-dotenv", True), ("tenacity", True), ("langgraph", True), ("langgraph-checkpoint-sqlite", True), ("crewai", False), ("google-adk", False), ("agent-framework", False))

def load_workflow_dataset() -> WorkflowDataset:
    dataset = WorkflowDataset.model_validate_json(DATA_PATH.read_text(encoding="utf-8"))
    ids = [c.case_id for c in dataset.cases]
    if len(dataset.cases) != 15 or len(ids) != len(set(ids)):
        raise ValueError("Expected exactly 15 canonical cases with unique IDs.")
    return dataset

def build_readiness() -> ReadinessReport:
    packages = []
    errors = []
    for package, required in PACKAGE_REQUIREMENTS:
        installed = installed_version(package)
        packages.append({"package":package, "required":required, "installed_version":installed, "ready":installed is not None})
        if required and installed is None:
            errors.append(f"Install missing package: {package}")
    credentials = [{"variable":"ANTHROPIC_API_KEY", "required":True, "configured":bool(os.getenv("ANTHROPIC_API_KEY"))}]
    if not credentials[0]["configured"]:
        errors.append("Set ANTHROPIC_API_KEY in Week2/.env")
    try:
        dataset = load_workflow_dataset()
        dataset_info = {"path":str(DATA_PATH), "exists":True, "case_count":len(dataset.cases), "unique_case_ids":True, "ready":True}
    except (OSError, ValueError) as error:
        dataset_info = {"path":str(DATA_PATH), "exists":DATA_PATH.is_file(), "case_count":0, "unique_case_ids":False, "ready":False, "error":str(error)}
        errors.append(f"Canonical dataset is not ready: {error}")
    return ReadinessReport(status="blocked" if errors else "ready", python_version=sys.version.split()[0], packages=packages, credentials=credentials, dataset=dataset_info, errors=errors)

readiness = build_readiness()
print(readiness.model_dump_json(indent=2))

In [ ]:
# AM2.2 — bounded structured provider call plus local grounding validation
class CaseBrief(BaseModel):
    model_config = ConfigDict(extra="forbid")
    case_id: str
    recommended_action: Literal["auto_approve", "refer_underwriter", "request_documents"]
    evidence_fields: list[Literal["product", "requested_limit_usd", "risk_flags", "documents_complete", "recommended_action"]] = Field(min_length=1)
    operator_summary: str

def validate_case_brief(brief: CaseBrief, source: WorkflowCase) -> None:
    if brief.case_id != source.case_id or brief.recommended_action != source.recommended_action:
        raise ValueError("Model output changed a canonical identity-critical field.")

if readiness.status != "ready":
    raise RuntimeError(f"AM2 provider boundary blocked by preflight: {readiness.errors}")
source_case = next((c for c in load_workflow_dataset().cases if c.case_id == CASE_ID), None)
if source_case is None:
    raise RuntimeError(f"Canonical case missing: {CASE_ID}")
started = time.perf_counter()
try:
    response = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=REQUEST_TIMEOUT_SECONDS, max_retries=1).messages.parse(
        model=MODEL, max_tokens=min(int(os.getenv("AGENT_MAX_TOKENS", "8000")), 1200), temperature=0,
        system="You are a read-only underwriting briefing assistant. Use only the supplied record and preserve case_id and recommended_action exactly.",
        messages=[{"role":"user", "content":f"Create a concise operator brief and list source fields used. Canonical record: {source_case.model_dump_json()}"}],
        output_format=CaseBrief,
    )
    if response.parsed_output is None:
        raise RuntimeError("Anthropic returned no parsed CaseBrief.")
    validate_case_brief(response.parsed_output, source_case)
    result = {"status":"success", "model":MODEL, "latency_seconds":round(time.perf_counter()-started,3), "stop_reason":response.stop_reason, "usage":response.usage.model_dump(), "result":response.parsed_output.model_dump()}
except (anthropic.APIError, RuntimeError, ValueError) as error:
    result = {"status":"error", "model":MODEL, "latency_seconds":round(time.perf_counter()-started,3), "error_type":type(error).__name__, "error_message":str(error)}
print(json.dumps({"stage":"AM2.2", "provider_run":result}, indent=2))
if result["status"] == "error":
    raise RuntimeError("AM2 provider call failed; inspect provider_run.")

## 03 Claude Agent SDK isolation primitives

Isolation has multiple dimensions: conversation context, filesystem working directory, inherited settings, model tools, MCP servers, network-capable built-ins, turns, and spend. A reusable skill adds instructions to the active context and is not an independent security boundary. An SDK subagent created with `AgentDefinition` receives fresh worker context and its own `tools`, `disallowedTools`, model, and `maxTurns`; the parent invokes it through the `Agent` tool and receives a bounded result. An agent team uses separately coordinated agents and peer communication, gaining autonomy at the cost of more messaging, state reconciliation, and failure modes.

`ClaudeAgentOptions` defines the parent boundary: `tools`/`allowed_tools`, named `agents`, `mcp_servers`, `strict_mcp_config`, `cwd`, `setting_sources`, `max_turns`, `max_budget_usd`, and structured output. `create_sdk_mcp_server` and `@tool` expose a narrow in-process domain operation. `can_use_tool` can inspect `ToolPermissionContext.agent_id`, allowing the canonical lookup only inside the delegated worker. Streaming `query()` messages exposes actual `SystemMessage`, `AssistantMessage`, tool-use/result blocks, and the final `ResultMessage` with session, turns, usage, cost, duration, and structured output.

The tradeoff is least privilege versus capability and latency. Denying broad built-ins and inherited settings makes the boundary inspectable, but every needed capability must be explicitly supplied. Delegation adds another model/tool turn and cost, so it is justified when context or capability isolation matters—not merely to rename a function call. An allowlist limits reachable tools; source-equality validation is still required to detect altered facts.

<img src="Images/d2am_t3_sdk_isolation.png" width="900" alt="Claude Agent SDK parent and isolated subagent boundaries for context, tools, permissions, settings, turns, and budget">

The first cell maps isolation primitives into genuine SDK configuration. The next runs the real in-process MCP worker with top-level `await`.


In [ ]:
# AM3.1 — isolation map and native SDK scope
import asyncio
from dataclasses import asdict
from typing import Any
from claude_agent_sdk import (
    AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, SystemMessage,
    ToolAnnotations, ToolResultBlock, ToolUseBlock, UserMessage, create_sdk_mcp_server, query, tool,
)
from claude_agent_sdk.types import PermissionResultAllow, PermissionResultDeny, ToolPermissionContext

CASE_LOOKUP_TOOL = "mcp__workflow_cases__case_lookup"
BUILTIN_TOOLS = ["Read", "Write", "Edit", "Bash", "Glob", "Grep", "WebSearch", "WebFetch"]

class IsolationBoundary(BaseModel):
    model_config = ConfigDict(extra="forbid")
    primitive: Literal["skill", "subagent", "agent_team"]
    context_boundary: str
    communication: str
    capability_boundary: str
    appropriate_when: str

class WorkerEvidence(BaseModel):
    model_config = ConfigDict(extra="forbid")
    worker_name: Literal["case_evidence_worker"]
    source_tool: Literal["case_lookup"]
    case: WorkflowCase
    operator_summary: str
    writes_performed: Literal[False]

isolation_boundaries = [
    IsolationBoundary(primitive="skill", context_boundary="Shared main-agent context", communication="Instructions load into active session", capability_boundary="No independent tool boundary", appropriate_when="Reusable guidance without a separate worker"),
    IsolationBoundary(primitive="subagent", context_boundary="Fresh worker context; summary returns", communication="Parent invokes Agent", capability_boundary="Own tools and disallowedTools", appropriate_when="Narrow task needs isolated context/capabilities"),
    IsolationBoundary(primitive="agent_team", context_boundary="Separate agent instances", communication="Peer messaging and task coordination", capability_boundary="Process-level coordination boundary", appropriate_when="Independent peers collaborate directly"),
]

def worker_definition() -> AgentDefinition:
    return AgentDefinition(description="Inspect exactly one canonical underwriting case.", prompt="Call case_lookup exactly once. Return every field exactly. Never infer, access files, or write.", tools=[CASE_LOOKUP_TOOL], disallowedTools=BUILTIN_TOOLS, model="inherit", maxTurns=3)

scope_preview = ClaudeAgentOptions(tools=["Agent"], allowed_tools=["Agent"], agents={"case_evidence_worker":worker_definition()}, cwd=SESSION_DIR, setting_sources=[], strict_mcp_config=True, max_turns=3)
print(json.dumps({"stage":"AM3.1", "isolation_map":[x.model_dump() for x in isolation_boundaries], "scope":{"cwd":str(scope_preview.cwd), "setting_sources":scope_preview.setting_sources, "strict_mcp_config":scope_preview.strict_mcp_config, "parent_tools":scope_preview.tools, "worker":asdict(scope_preview.agents["case_evidence_worker"])}}, default=str, indent=2))

In [ ]:
# AM3.2 — actual delegated worker with one in-process MCP read tool
@tool(
    "case_lookup", "Return exactly one canonical underwriting case by UW identifier.",
    {"type":"object", "properties":{"case_id":{"type":"string", "pattern":"^UW-[0-9]{3}$"}}, "required":["case_id"], "additionalProperties":False},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def case_lookup(args: dict[str, Any]) -> dict[str, Any]:
    record = next((c for c in load_workflow_dataset().cases if c.case_id == str(args["case_id"])), None)
    payload = {"found":False, "case_id":args["case_id"]} if record is None else {"found":True, **record.model_dump()}
    return {"content":[{"type":"text", "text":json.dumps(payload)}]}

def expected_worker_summary(source: WorkflowCase) -> str:
    flags = ",".join(source.risk_flags) if source.risk_flags else "none"
    return f"{source.case_id} | {source.applicant} | {source.product} | limit_usd={source.requested_limit_usd} | risk_flags={flags} | documents_complete={str(source.documents_complete).lower()} | recommended_action={source.recommended_action}"

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("AM3 preflight blocked: set ANTHROPIC_API_KEY in Week2/.env.")
source = next((c for c in load_workflow_dataset().cases if c.case_id == CASE_ID), None)
if source is None:
    raise RuntimeError(f"Canonical case missing: {CASE_ID}")
server = create_sdk_mcp_server(name="workflow_cases", version="1.0.0", tools=[case_lookup])
permission_events: list[dict] = []

async def permission_gate(tool_name: str, input_data: dict[str, Any], context: ToolPermissionContext):
    allowed = tool_name == CASE_LOOKUP_TOOL and bool(context.agent_id)
    permission_events.append({"tool":tool_name, "subagent_context":bool(context.agent_id), "allowed":allowed})
    return PermissionResultAllow(updated_input=input_data) if allowed else PermissionResultDeny(message="Only the isolated worker may call case_lookup.", interrupt=False)

options = ClaudeAgentOptions(
    model=MODEL, tools=["Agent"], allowed_tools=["Agent", "StructuredOutput"], can_use_tool=permission_gate,
    agents={"case_evidence_worker":worker_definition()}, mcp_servers={"workflow_cases":server}, strict_mcp_config=True,
    output_format={"type":"json_schema", "schema":WorkerEvidence.model_json_schema()},
    max_turns=min(int(os.getenv("AGENT_MAX_STEPS", "10")), 5), max_budget_usd=float(os.getenv("AGENT_MAX_BUDGET_USD", "0.50")),
    cwd=SESSION_DIR, setting_sources=[],
)
async def prompt_stream(text: str):
    yield {"type":"user", "message":{"role":"user", "content":text}}

prompt = f"Use Agent to invoke case_evidence_worker for {CASE_ID}. Return WorkerEvidence, copy canonical fields exactly, source_tool=case_lookup, writes_performed=false, and operator_summary exactly: {expected_worker_summary(source)}"
events = []
result = None
delegated = False
started = time.perf_counter()
timeout = max(120.0, REQUEST_TIMEOUT_SECONDS * 2)
async with asyncio.timeout(timeout):
    async for message in query(prompt=prompt_stream(prompt), options=options):
        if isinstance(message, SystemMessage):
            events.append({"type":"SystemMessage", "subtype":message.subtype})
        elif isinstance(message, AssistantMessage):
            uses = [{"name":b.name, "input":b.input} for b in message.content if isinstance(b, ToolUseBlock)]
            delegated = delegated or any(x["name"] == "Agent" for x in uses)
            events.append({"type":"AssistantMessage", "model":message.model, "tool_uses":uses})
        elif isinstance(message, UserMessage):
            events.extend({"type":"ToolResultBlock", "tool_use_id":b.tool_use_id, "is_error":b.is_error} for b in message.content if isinstance(b, ToolResultBlock))
        elif isinstance(message, ResultMessage):
            result = message
if result is None or result.is_error or not delegated:
    raise RuntimeError("AM3 SDK run did not complete through the delegated worker boundary.")
report = WorkerEvidence.model_validate(result.structured_output)
if report.case != source or report.operator_summary != expected_worker_summary(source):
    raise ValueError("Worker report differs from canonical source evidence.")
print(json.dumps({"stage":"AM3.2", "native_events":events, "session_id":result.session_id, "turns":result.num_turns, "usage":result.usage, "cost_usd":result.total_cost_usd, "latency_seconds":round(time.perf_counter()-started,3), "permission_events":permission_events, "structured_output":report.model_dump()}, default=str, indent=2))

## 04 Lifecycle hooks as deterministic policy

Claude Agent SDK hooks run application code at named lifecycle events. `HookMatcher` registers callbacks under `PreToolUse`, `PostToolUse`, `PostToolUseFailure`, and `Stop`. A `PreToolUse` callback can return `hookSpecificOutput` with `permissionDecision="allow"` or `"deny"`; this is the correct place to validate tool name, input shape, identifier policy, and call count before execution. Post-success and post-failure hooks record what actually happened, while `Stop` marks termination. Correlating events with `tool_use_id` and monotonic sequence evidence distinguishes requested, executed, failed, and completed operations.

Hooks complement rather than replace tool allowlists and SDK permission callbacks. An allowlist limits which tool can be proposed; the pre-hook evaluates each proposed invocation; the domain tool enforces data rules; post-hooks create evidence. Hooks should fail closed and must not mutate a denial into apparent success. Persisted audit data must be redacted and access-controlled because tool inputs and responses can contain sensitive records.

The tradeoff is centralized policy and observability versus coupling and overhead. Hooks provide consistent controls across model-selected calls, but incorrect matchers or permissive callbacks affect every matching operation. `setting_sources=["project"]` may load project configuration and is convenient for class, while an empty setting source is a tighter isolation choice. Appending JSONL gives durable inspection but requires retention, concurrency, and privacy controls in production.

<img src="Images/d2am_t4_lifecycle_hooks.png" width="900" alt="Claude Agent SDK lifecycle hooks enforcing policy before tools and recording success, failure, and stop evidence">

This progression starts with observable order and then applies strict shape/count checks. Durable JSONL persistence remains explicitly opt-in.


In [ ]:
# AM4.1 — ordered, strict lifecycle hooks
import asyncio
from claude_agent_sdk import HookMatcher

audit: list[dict[str, Any]] = []

def append_audit_event(event: str, input_data: dict[str, Any], tool_use_id: str | None) -> None:
    audit.append({"sequence":len(audit)+1, "event":event, "tool_name":input_data.get("tool_name"), "tool_use_id":tool_use_id, "tool_input":input_data.get("tool_input"), "tool_response":input_data.get("tool_response"), "error":input_data.get("error"), "monotonic_ns":time.monotonic_ns()})

async def pre_tool_use(input_data: dict[str, Any], tool_use_id: str | None, context: dict[str, Any]) -> dict[str, Any]:
    del context
    append_audit_event("PreToolUse", input_data, tool_use_id)
    tool_input = input_data.get("tool_input", {})
    allowed = input_data.get("tool_name") == "mcp__underwriting__case_lookup" and isinstance(tool_input, dict) and set(tool_input) == {"case_id"} and isinstance(tool_input.get("case_id"), str) and sum(x["event"] == "PreToolUse" for x in audit) == 1
    return {"hookSpecificOutput":{"hookEventName":"PreToolUse", "permissionDecision":"allow" if allowed else "deny", "permissionDecisionReason":"Validated one canonical read-only lookup." if allowed else "Denied malformed, unexpected, or repeated tool call."}}

async def post_tool_use(input_data, tool_use_id, context):
    del context
    append_audit_event("PostToolUse", input_data, tool_use_id)
    return {}

async def post_tool_failure(input_data, tool_use_id, context):
    del context
    append_audit_event("PostToolUseFailure", input_data, tool_use_id)
    return {}

async def stop_hook(input_data, tool_use_id, context):
    del context
    append_audit_event("Stop", input_data, tool_use_id)
    return {}

@tool("case_lookup", "Read one canonical underwriting case by case_id.", {"case_id":str}, annotations=ToolAnnotations(readOnlyHint=True))
async def case_lookup(args: dict[str, Any]) -> dict[str, Any]:
    case = next((c for c in load_workflow_dataset().cases if c.case_id == str(args["case_id"])), None)
    if case is None:
        raise LookupError(f"No canonical workflow case exists for {args['case_id']}.")
    return {"content":[{"type":"text", "text":case.model_dump_json()}]}

print({"stage":"AM4.1", "registered_hooks":["PreToolUse", "PostToolUse", "PostToolUseFailure", "Stop"], "policy":"one exact read-only case_lookup"})

In [ ]:
# AM4.2 — real hooked call; audit persistence is local-write gated
PERSIST_HOOK_AUDIT = False  # Set True only when you explicitly want a local JSONL write.
audit.clear()
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("AM4 preflight blocked: set ANTHROPIC_API_KEY in Week2/.env.")
server = create_sdk_mcp_server(name="underwriting", version="1.0.0", tools=[case_lookup])
options = ClaudeAgentOptions(
    model=MODEL, tools=[], mcp_servers={"underwriting":server}, allowed_tools=["mcp__underwriting__case_lookup"],
    hooks={"PreToolUse":[HookMatcher(matcher=None, hooks=[pre_tool_use])], "PostToolUse":[HookMatcher(matcher=None, hooks=[post_tool_use])], "PostToolUseFailure":[HookMatcher(matcher=None, hooks=[post_tool_failure])], "Stop":[HookMatcher(matcher=None, hooks=[stop_hook])]},
    max_turns=min(int(os.getenv("AGENT_MAX_STEPS", "6")), 6), max_budget_usd=float(os.getenv("AGENT_MAX_BUDGET_USD", "0.50")), cwd=SESSION_DIR, setting_sources=[], strict_mcp_config=True,
)
result = None
async with asyncio.timeout(REQUEST_TIMEOUT_SECONDS):
    async for message in query(prompt=f"Call case_lookup exactly once for {CASE_ID}. Use only its result; never invent a case.", options=options):
        if isinstance(message, ResultMessage):
            result = message
if result is None or result.is_error:
    raise RuntimeError("AM4 Agent SDK run did not complete successfully.")
audit_path = OUTPUT_DIR / "topic4_hook_audit.jsonl"
if PERSIST_HOOK_AUDIT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with audit_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps({"session_id":result.session_id, "events":audit}) + "\n")
print(json.dumps({"stage":"AM4.2", "audit_persisted":PERSIST_HOOK_AUDIT, "audit_path":str(audit_path) if PERSIST_HOOK_AUDIT else None, "events":audit, "result":result.result, "session_id":result.session_id}, default=str, indent=2))

## 05 Reliability engineering for external providers

Reliability begins by classifying a real exception before choosing policy. Timeouts, connection failures, rate limits, selected conflict/timeout statuses, and provider 5xx responses may be transient. Authentication, permission, bad request, missing model, schema, and unknown failures should fail fast until credentials, access, input, or code changes. Retries need capped attempts and randomized exponential backoff to avoid retry storms; the provider client's own retries should be disabled when Tenacity owns the policy so limits remain observable.

A fallback is another genuinely configured model, not locally invented output. Each model receives its own bounded retry sequence. Run evidence should retain model, attempt number, latency, status, token usage, request ID, failure type/classification, selected model, configured limits, and final text. A timeout limits request duration; `max_tokens` limits response size; attempt count limits requests. A configured USD budget is useful operational metadata here, but direct Anthropic Messages calls do not enforce that value automatically—cost enforcement requires pricing-aware accounting or a runtime that implements a budget cap.

The tradeoff is resilience versus latency, cost, and duplicate effects. Retries improve recovery from transient read/model calls but multiply worst-case time and spend. Write operations need idempotency before any retry. A model fallback may restore availability but can change quality, behavior, access, and pricing, so degraded behavior must be explicit and evaluated.

<img src="Images/d2am_t5_reliability_engineering.png" width="900" alt="Reliability flow classifying external provider failures before bounded retry, fallback, or fail-fast handling">

The first cell defines the failure taxonomy. The next performs genuine provider attempts and uses a fallback only when `ANTHROPIC_FALLBACK_MODEL` names a distinct configured model.


In [ ]:
# AM5.1 — deterministic provider-failure taxonomy
from dataclasses import dataclass, field, asdict
from tenacity import AsyncRetrying, retry_if_exception, stop_after_attempt, wait_random_exponential

@dataclass(frozen=True)
class FailureClassification:
    disposition: Literal["retryable", "nonretryable"]
    category: str
    reason: str

def classify_failure(error: BaseException) -> FailureClassification:
    if isinstance(error, (anthropic.APITimeoutError, anthropic.APIConnectionError)):
        return FailureClassification("retryable", "transport", "Connection and timeout failures may be transient.")
    if isinstance(error, anthropic.RateLimitError):
        return FailureClassification("retryable", "rate_limit", "Provider rate limits may clear after backoff.")
    if isinstance(error, anthropic.InternalServerError):
        return FailureClassification("retryable", "provider_5xx", "Provider server failures may be transient.")
    if isinstance(error, (anthropic.AuthenticationError, anthropic.PermissionDeniedError, anthropic.BadRequestError, anthropic.NotFoundError)):
        return FailureClassification("nonretryable", "request_or_access", "Credentials, access, model, or request must be corrected.")
    if isinstance(error, anthropic.APIStatusError):
        status = error.status_code
        return FailureClassification("retryable", f"http_{status}", "Transient HTTP status.") if status in {408,409,429} or status >= 500 else FailureClassification("nonretryable", f"http_{status}", "Permanent HTTP status for this request.")
    return FailureClassification("nonretryable", type(error).__name__, "Unknown failures fail closed.")

def should_retry(error: BaseException) -> bool:
    return classify_failure(error).disposition == "retryable"

print({"stage":"AM5.1", "retryable":["timeout", "connection", "rate_limit", "408", "409", "429", "5xx"], "nonretryable":["authentication", "permission", "bad_request", "not_found", "unknown"]})

In [ ]:
# AM5.2 — bounded selective retry and genuinely configured model fallback
MAX_ATTEMPTS_PER_MODEL = 3
MAX_OUTPUT_TOKENS = 96
RELIABILITY_PROMPT = "Reply with exactly: bounded reliable workflow completed"
primary_model = MODEL.strip()
fallback_model = os.getenv("ANTHROPIC_FALLBACK_MODEL", "").strip()
configured_models = [primary_model] + ([fallback_model] if fallback_model and fallback_model != primary_model else [])
configured_usd_budget = float(os.getenv("AGENT_MAX_BUDGET_USD", "0.50"))
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("AM5 preflight blocked: set ANTHROPIC_API_KEY in Week2/.env.")
if not 1 <= MAX_ATTEMPTS_PER_MODEL <= 5 or not 1 <= MAX_OUTPUT_TOKENS <= 512:
    raise ValueError("Configured reliability bounds are outside teaching limits.")

attempts = []
selected_model = None
final_text = None
last_error = None
client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=REQUEST_TIMEOUT_SECONDS, max_retries=0)
started = time.perf_counter()
try:
    for candidate_model in configured_models:
        try:
            async for attempt in AsyncRetrying(stop=stop_after_attempt(MAX_ATTEMPTS_PER_MODEL), wait=wait_random_exponential(multiplier=1, max=8), retry=retry_if_exception(should_retry), reraise=True):
                with attempt:
                    attempt_started = time.perf_counter()
                    try:
                        response = await client.messages.create(model=candidate_model, max_tokens=MAX_OUTPUT_TOKENS, messages=[{"role":"user", "content":RELIABILITY_PROMPT}])
                    except Exception as error:
                        attempts.append({"model":candidate_model, "attempt":attempt.retry_state.attempt_number, "latency_ms":round((time.perf_counter()-attempt_started)*1000,2), "status":"failed", "request_id":getattr(error,"request_id",None), "failure_type":type(error).__name__, "classification":asdict(classify_failure(error))})
                        raise
                    attempts.append({"model":candidate_model, "attempt":attempt.retry_state.attempt_number, "latency_ms":round((time.perf_counter()-attempt_started)*1000,2), "status":"success", "input_tokens":response.usage.input_tokens, "output_tokens":response.usage.output_tokens, "request_id":getattr(response,"_request_id",None)})
            selected_model = candidate_model
            final_text = "".join(block.text for block in response.content if block.type == "text")
            break
        except Exception as error:
            last_error = error
finally:
    await client.close()
result = {
    "stage":"AM5.2", "status":"complete" if selected_model else "failed", "selected_model":selected_model,
    "configured_models":configured_models, "fallback_configured":len(configured_models)>1,
    "max_attempts_per_model":MAX_ATTEMPTS_PER_MODEL, "max_output_tokens_per_attempt":MAX_OUTPUT_TOKENS,
    "max_requests":len(configured_models)*MAX_ATTEMPTS_PER_MODEL, "request_timeout_seconds":REQUEST_TIMEOUT_SECONDS,
    "configured_usd_budget":configured_usd_budget, "usd_budget_enforced_by_direct_messages_api":False,
    "attempts":attempts, "total_input_tokens":sum(x.get("input_tokens",0) for x in attempts), "total_output_tokens":sum(x.get("output_tokens",0) for x in attempts),
    "total_latency_ms":round((time.perf_counter()-started)*1000,2), "text":final_text,
}
print(json.dumps(result, indent=2))
if selected_model is None:
    raise RuntimeError("All genuinely configured models failed; no fallback output was fabricated.") from last_error

## 06 Checkpointing, HITL, and idempotent action

Checkpointing, human approval, and safe execution are separate guarantees. A LangGraph `StateGraph` defines typed shared state and explicit nodes/edges. Compiling with `SqliteSaver` persists thread-scoped checkpoints; every later `get_state` or resume must use the same `thread_id`. Persistence enables recovery but does not imply that a person approved anything.

`interrupt(payload)` pauses inside a node and exposes the exact proposal to an external reviewer. `Command(resume=value)` continues that same thread with an explicit decision object. The graph must validate decision and actor, persist the record, and keep `action_executed=False` while merely proposing or approving. Because an interrupted node may be replayed, code before and after the interrupt must be designed with replay semantics in mind.

Crossing a write boundary requires both recorded learner approval and a nonempty idempotency key. A durable action ledger with a unique key returns the original receipt on replay and rejects key reuse for a different case/action. This yields at-most-once logical execution for this local ledger. The tradeoff is additional storage and reconciliation complexity: SQLite is inspectable for a single-machine lab, but production needs transactional domain-side idempotency, authenticated approvals, durable shared storage, concurrency handling, audit retention, and authorization at the tool itself.

<img src="Images/d2am_t6_orchestration_hitl.png" width="900" alt="Human-in-the-loop state flow separating checkpoint persistence from recorded approval or rejection">

The cells below define one progressive graph: canonical proposal → SQLite checkpoint and interrupt → explicit resume → approval-gated idempotent ledger. All local writes are disabled by default; the default decision is reject.


In [ ]:
# AM6.1 — explicit local-write and human-decision controls
ENABLE_CHECKPOINT_WRITES = False  # Set True to permit local SQLite checkpoint writes.
ENABLE_ACTION_LEDGER_WRITES = False  # Set True separately to permit the local action ledger write.
HITL_DECISION: Literal["approve", "reject"] = "reject"  # Safe default.
HITL_ACTOR = "learner"
LEARNER_IDEMPOTENCY_KEY = ""  # For approval, supply a fresh nonempty key yourself; never auto-generate it.
THREAD_ID = "uw-idempotent-hitl-notebook"

print({"stage":"AM6.1", "checkpoint_writes_enabled":ENABLE_CHECKPOINT_WRITES, "ledger_writes_enabled":ENABLE_ACTION_LEDGER_WRITES, "default_decision":HITL_DECISION, "idempotency_key_supplied":bool(LEARNER_IDEMPOTENCY_KEY.strip())})

In [ ]:
# AM6.2 — checkpointed HITL graph and approval validation
import sqlite3
from typing import TypedDict
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

CHECKPOINT_DB = OUTPUT_DIR / "topic6_idempotent_hitl.sqlite"
ACTION_DB = OUTPUT_DIR / "topic6_action_ledger.sqlite"

class WorkflowState(TypedDict, total=False):
    case_id: str
    case: dict[str, Any]
    proposed_action: str
    stage: Literal["loaded", "proposed", "approved", "rejected", "complete"]
    approval: dict[str, Any] | None
    idempotency_key: str | None
    action_executed: bool
    action_receipt: dict[str, Any] | None
    evidence: list[dict[str, Any]]

def load_case(state: WorkflowState) -> WorkflowState:
    case = next((c for c in load_workflow_dataset().cases if c.case_id == state["case_id"]), None)
    if case is None:
        raise LookupError(f"No canonical workflow case exists for {state['case_id']}.")
    return {"case":case.model_dump(), "stage":"loaded", "approval":None, "idempotency_key":None, "action_executed":False, "action_receipt":None, "evidence":[{"event":"case_loaded", "case_id":state["case_id"]}]}

def propose_action(state: WorkflowState) -> WorkflowState:
    action = str(state["case"]["recommended_action"])
    return {"proposed_action":action, "stage":"proposed", "evidence":[*state["evidence"], {"event":"action_proposed", "action":action, "executed":False}]}

def require_approval(state: WorkflowState) -> WorkflowState:
    decision = interrupt({"question":"Approve this proposed underwriting action?", "case_id":state["case_id"], "proposed_action":state["proposed_action"], "options":["approve","reject"], "approval_requires":["actor","idempotency_key"], "action_executed":False})
    if not isinstance(decision, dict):
        raise ValueError("Resume value must be an approval object.")
    choice, actor, key = decision.get("decision"), decision.get("actor"), decision.get("idempotency_key")
    if choice not in {"approve","reject"} or not isinstance(actor, str) or not actor.strip():
        raise ValueError("Resume requires decision=approve|reject and a nonempty actor.")
    if choice == "approve" and (not isinstance(key, str) or not key.strip()):
        raise ValueError("Approval requires a learner-supplied fresh idempotency key.")
    approved = choice == "approve"
    approval = {"approved":approved, "actor":actor.strip()}
    return {"stage":"approved" if approved else "rejected", "approval":approval, "idempotency_key":key.strip() if approved else None, "action_executed":False, "evidence":[*state["evidence"], {"event":"human_decision", **approval, "idempotency_key":key.strip() if approved else None, "action_executed":False}]}

print({"stage":"AM6.2", "nodes":["load_case", "propose_action", "require_approval", "execute_if_approved"], "checkpoint_db":str(CHECKPOINT_DB)})

In [ ]:
# AM6.3 — durable idempotency ledger and approval-gated action node
def initialize_action_ledger(connection: sqlite3.Connection) -> None:
    connection.execute("""CREATE TABLE IF NOT EXISTS action_receipts (idempotency_key TEXT PRIMARY KEY, case_id TEXT NOT NULL, proposed_action TEXT NOT NULL, actor TEXT NOT NULL, receipt_json TEXT NOT NULL)""")

def record_action_once(case_id: str, proposed_action: str, actor: str, idempotency_key: str) -> dict[str, Any]:
    if not ENABLE_ACTION_LEDGER_WRITES:
        raise PermissionError("Local action ledger writes are disabled. Set ENABLE_ACTION_LEDGER_WRITES=True explicitly.")
    with sqlite3.connect(ACTION_DB) as connection:
        initialize_action_ledger(connection)
        existing = connection.execute("SELECT case_id, proposed_action, receipt_json FROM action_receipts WHERE idempotency_key = ?", (idempotency_key,)).fetchone()
        if existing is not None:
            if existing[0] != case_id or existing[1] != proposed_action:
                raise RuntimeError("Idempotency key collision: existing action payload differs.")
            return {**json.loads(existing[2]), "executed":False, "duplicate_replay":True}
        receipt = {"case_id":case_id, "action":proposed_action, "actor":actor, "idempotency_key":idempotency_key, "executed":True, "duplicate_replay":False}
        connection.execute("INSERT INTO action_receipts (idempotency_key, case_id, proposed_action, actor, receipt_json) VALUES (?, ?, ?, ?, ?)", (idempotency_key, case_id, proposed_action, actor, json.dumps(receipt)))
        return receipt

def execute_if_approved(state: WorkflowState) -> WorkflowState:
    approval = state.get("approval") or {}
    if not approval.get("approved"):
        return {"stage":"complete", "action_executed":False, "action_receipt":None, "evidence":[*state["evidence"], {"event":"action_skipped", "reason":"not_approved"}]}
    key = state.get("idempotency_key")
    if not key:
        raise RuntimeError("Approved state is missing its learner-supplied idempotency key.")
    receipt = record_action_once(state["case_id"], state["proposed_action"], str(approval["actor"]), key)
    return {"stage":"complete", "action_executed":bool(receipt["executed"]), "action_receipt":receipt, "evidence":[*state["evidence"], {"event":"action_result", "executed":receipt["executed"], "duplicate_replay":receipt["duplicate_replay"], "idempotency_key":key}]}

def build_graph() -> StateGraph:
    builder = StateGraph(WorkflowState)
    builder.add_node("load_case", load_case)
    builder.add_node("propose_action", propose_action)
    builder.add_node("require_approval", require_approval)
    builder.add_node("execute_if_approved", execute_if_approved)
    builder.add_edge(START, "load_case")
    builder.add_edge("load_case", "propose_action")
    builder.add_edge("propose_action", "require_approval")
    builder.add_edge("require_approval", "execute_if_approved")
    builder.add_edge("execute_if_approved", END)
    return builder

print({"stage":"AM6.3", "ledger":str(ACTION_DB), "write_guard":ENABLE_ACTION_LEDGER_WRITES, "collision_policy":"reject changed payload; return original receipt on exact replay"})

In [ ]:
# AM6.4 — start: persist proposal and pause before any action
if not ENABLE_CHECKPOINT_WRITES:
    print({"stage":"AM6.4", "status":"skipped", "reason":"Set ENABLE_CHECKPOINT_WRITES=True to create local SQLite checkpoints."})
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    config = {"configurable":{"thread_id":THREAD_ID}}
    with SqliteSaver.from_conn_string(str(CHECKPOINT_DB)) as checkpointer:
        graph = build_graph().compile(checkpointer=checkpointer)
        result = graph.invoke({"case_id":CASE_ID}, config=config)
        snapshot = graph.get_state(config)
        print(json.dumps({"stage":"AM6.4", "thread_id":THREAD_ID, "next_nodes":list(snapshot.next), "interrupts":result.get("__interrupt__", []), "state":snapshot.values}, default=str, indent=2))

In [ ]:
# AM6.5 — resume: reject by default; approval needs both write flags and a fresh learner key
if not ENABLE_CHECKPOINT_WRITES:
    print({"stage":"AM6.5", "status":"skipped", "decision":HITL_DECISION, "reason":"Checkpoint writes are disabled."})
else:
    if HITL_DECISION == "approve":
        if not ENABLE_ACTION_LEDGER_WRITES:
            raise PermissionError("Approval cannot execute while ENABLE_ACTION_LEDGER_WRITES=False.")
        if not LEARNER_IDEMPOTENCY_KEY.strip():
            raise ValueError("Approval requires a learner-supplied fresh IDEMPOTENCY_KEY.")
    config = {"configurable":{"thread_id":THREAD_ID}}
    resume_payload = {"decision":HITL_DECISION, "actor":HITL_ACTOR, "idempotency_key":LEARNER_IDEMPOTENCY_KEY if HITL_DECISION == "approve" else None}
    with SqliteSaver.from_conn_string(str(CHECKPOINT_DB)) as checkpointer:
        graph = build_graph().compile(checkpointer=checkpointer)
        if not graph.get_state(config).values:
            raise RuntimeError("No paused checkpoint exists. Run AM6.4 first with the same THREAD_ID.")
        result = graph.invoke(Command(resume=resume_payload), config=config)
        snapshot = graph.get_state(config)
        print(json.dumps({"stage":"AM6.5", "thread_id":THREAD_ID, "decision":HITL_DECISION, "next_nodes":list(snapshot.next), "state":result}, default=str, indent=2))

## Exercises

Two short apply tasks after §06. Attempt them here before opening [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb). Run this notebook from the top through §06 first so helpers like `build_readiness`, `validate_case_brief`, `CaseBrief`, and the HITL flags exist in the kernel.

### Exercise AM-1 — Reject a grounded CaseBrief mismatch

**Concept:** Pydantic structured output guarantees shape, not factual truth. Local grounding must reject a brief whose `case_id` or `recommended_action` does not match the canonical source.

**Task:** Build (or mutate) a `CaseBrief` whose `case_id` does **not** match the canonical `UW-001` source case, then call `validate_case_brief` and catch the failure. Also show what `build_readiness()` reports when the dataset path is temporarily wrong (or invent a failing readiness path without inventing domain records).

**Expected different result:** validation raises (or returns a clear failure) instead of accepting a schema-valid but ungrounded brief.

**TODO hints (no code):**
- TODO: load the real `UW-001` source from `load_workflow_dataset()`
- TODO: construct a `CaseBrief` with a wrong `case_id` (for example `UW-999`) but otherwise plausible fields
- TODO: call `validate_case_brief(brief, source)` inside try/except and display the error
- TODO: optionally break readiness (wrong path) and print `status != ready`


In [ ]:
# Exercise AM-1 workspace
# TODO: construct an ungrounded CaseBrief and show validate_case_brief rejecting it
# TODO: optionally demonstrate readiness failing when the dataset is not ready

raise NotImplementedError("Exercise AM-1: reject an ungrounded CaseBrief yourself")


### Exercise AM-2 — Approve HITL without an idempotency key

**Concept:** Crossing a write boundary needs recorded learner approval **and** a nonempty idempotency key. Approval alone must not execute the action.

**Task:** Using the §06 HITL controls (`HITL_DECISION`, `LEARNER_IDEMPOTENCY_KEY`, `ENABLE_ACTION_LEDGER_WRITES`), set `HITL_DECISION = "approve"` but leave `LEARNER_IDEMPOTENCY_KEY` empty (or whitespace). Show that the resume path refuses to execute (raises `ValueError` / equivalent) when the key is missing.

**Expected different result:** approval without a key fails closed; no action receipt is written.

**TODO hints (no code):**
- TODO: reuse AM6.1 flags / AM6.5 resume logic patterns from above
- TODO: set decision to approve with an empty key
- TODO: catch and display the refusal (do not invent a successful ledger write)
- TODO: optionally contrast with reject (no key required)


In [ ]:
# Exercise AM-2 workspace
# TODO: attempt approve resume with an empty LEARNER_IDEMPOTENCY_KEY and show fail-closed behavior

raise NotImplementedError("Exercise AM-2: prove approval without an idempotency key is blocked")


### Operator checklist

- Choose frameworks from auditable API evidence, not feature-count marketing.
- Fail-fast readiness before paid provider calls; validate identity-critical fields after structured output.
- Isolate subagents with explicit tools, permissions, turns, and budgets.
- Use lifecycle hooks for deterministic allow/deny and correlated audit evidence.
- Retry only classified transient failures; fall back to a genuinely configured model.
- Checkpoint ≠ approval; approval + idempotency key precede any write/action.
